In [0]:
failures = spark.table("workspace.default.silver_failures")

failures.select("failure").distinct().show(truncate=False)

+-------+
|failure|
+-------+
|comp1  |
|comp2  |
|comp3  |
|comp4  |
+-------+



In [0]:
from pyspark.sql.functions import col, count, when, expr
errors = spark.table("workspace.default.silver_errors").alias("e")
failures = spark.table("workspace.default.silver_failures").alias("f")

error_failure = (
    errors.join(
        failures,
        (col("e.machineID") == col("f.machineID")) &
        (col("e.datetime") < col("f.datetime")) &
        (col("e.datetime") >= col("f.datetime") - expr("INTERVAL 48 HOURS")) &
        (col("e.datetime") <= col("f.datetime") - expr("INTERVAL 24 HOURS")),
        "inner"
    )
)

result = (
    error_failure
    .groupBy("f.failure")
    .agg(
        count("*").alias("errors_24_to_48h_before_failure")
    )
    .orderBy("f.failure")
)

result.show()

+-------+-------------------------------+
|failure|errors_24_to_48h_before_failure|
+-------+-------------------------------+
|  comp1|                            223|
|  comp2|                            561|
|  comp3|                            156|
|  comp4|                            232|
+-------+-------------------------------+



In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.gold_q1_error_failure"

if not spark.catalog.tableExists(target_table):

    result.write.format("delta").saveAsTable(target_table)

else:

    target = DeltaTable.forName(spark, target_table)

    target.alias("t").merge(
        result.alias("s"),
        "t.failure = s.failure"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print("Gold Q1 table created/updated")
result.printSchema()

Gold Q1 table created/updated
root
 |-- failure: string (nullable = true)
 |-- errors_24_to_48h_before_failure: long (nullable = false)



In [0]:
gold_q1 = spark.table("workspace.default.gold_q1_error_failure")

gold_q1.show()

+-------+-------------------------------+
|failure|errors_24_to_48h_before_failure|
+-------+-------------------------------+
|  comp1|                            223|
|  comp2|                            561|
|  comp3|                            156|
|  comp4|                            232|
+-------+-------------------------------+



In [0]:
display(gold_q1)

failure,errors_24_to_48h_before_failure
comp1,223
comp2,561
comp3,156
comp4,232


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, count

error_id_analysis = (
    error_failure
    .groupBy("f.failure", "e.errorID")
    .agg(
        count("*").alias("error_count_24_to_48h")
    )
    .orderBy("f.failure", col("error_count_24_to_48h").desc())
)

error_id_analysis.show(50, truncate=False)

+-------+-------+---------------------+
|failure|errorID|error_count_24_to_48h|
+-------+-------+---------------------+
|comp1  |error1 |188                  |
|comp1  |error3 |12                   |
|comp1  |error2 |10                   |
|comp1  |error4 |7                    |
|comp1  |error5 |6                    |
|comp2  |error3 |263                  |
|comp2  |error2 |259                  |
|comp2  |error1 |17                   |
|comp2  |error5 |14                   |
|comp2  |error4 |8                    |
|comp3  |error4 |128                  |
|comp3  |error3 |9                    |
|comp3  |error5 |7                    |
|comp3  |error2 |6                    |
|comp3  |error1 |6                    |
|comp4  |error5 |177                  |
|comp4  |error2 |19                   |
|comp4  |error3 |18                   |
|comp4  |error1 |13                   |
|comp4  |error4 |5                    |
+-------+-------+---------------------+



In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.gold_q1_error_id_failure"

if not spark.catalog.tableExists(target_table):

    error_id_analysis.write.format("delta").saveAsTable(target_table)

else:

    target = DeltaTable.forName(spark, target_table)

    target.alias("t").merge(
        error_id_analysis.alias("s"),
        "t.failure = s.failure AND t.errorID = s.errorID"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print("Gold Q1 error ID analysis table created/updated")
error_id_analysis.printSchema()

Gold Q1 error ID analysis table created/updated
root
 |-- failure: string (nullable = true)
 |-- errorID: string (nullable = true)
 |-- error_count_24_to_48h: long (nullable = false)



In [0]:
display(error_id_analysis)

failure,errorID,error_count_24_to_48h
comp1,error1,188
comp1,error3,12
comp1,error2,10
comp1,error4,7
comp1,error5,6
comp2,error3,263
comp2,error2,259
comp2,error1,17
comp2,error5,14
comp2,error4,8


Databricks visualization. Run in Databricks to view.